# Session 2 lab · Image formation & sensing

One notebook for the whole session, one section per part of the slides:

| Part | You will |
|---|---|
| **A · Pinhole** | project a 3D cube with `x = f·X/Z`, and watch parallel rails meet |
| **B · Lenses** | compute fields of view, zoom by cropping, and plot blur circles |
| **C · Sensors** | treat an image as an array: resolution, bit depth, shot noise, dynamic range |
| **D · Colour** | split RGB, build a Bayer mosaic, demosaic it, and rotate hues |

**How to work:** run the *Guided* cells in order and read their output. In each **Your turn** cell, replace `None` with your answer, run it, and `check()` prints ✓ or ✗. Everything runs on free Colab — no files to upload.

In [ ]:
# Setup — run this first
import numpy as np, matplotlib.pyplot as plt, cv2
from skimage import data            # sample images that ship with scikit-image (already on Colab)
plt.rcParams['figure.dpi'] = 110
rng = np.random.default_rng(0)

def check(name, value, expected, tol=0.02):
    """Compare your answer with the expected value (relative tolerance)."""
    if value is None:
        print(f'{name}: not answered yet — replace None with your answer'); return
    ok = abs(float(value) - expected) <= tol * max(1, abs(expected))
    print(f'{name}: {value:.4g} → ' + ('✓ correct' if ok else f'✗ not yet (expected about {expected:.4g})'))

def show(images, titles, cmap='gray', size=3.2):
    fig, ax = plt.subplots(1, len(images), figsize=(size * len(images), size))
    for a, im, t in zip(np.atleast_1d(ax), images, titles):
        a.imshow(im, cmap=cmap, vmin=0, vmax=255 if im.dtype == np.uint8 else None); a.set_title(t); a.axis('off')
    plt.tight_layout(); plt.show()

astronaut = data.astronaut()        # 512 × 512 × 3, uint8 — a NASA photo, public domain
print('ready · OpenCV', cv2.__version__)

## A · Pinhole & perspective

A pinhole camera maps a 3D point $(X, Y, Z)$ to the image point
$$x = f\,\frac{X}{Z}, \qquad y = f\,\frac{Y}{Z}$$
with $f$ in millimetres and $X, Y, Z$ in metres — the metres cancel, so $x, y$ come out in millimetres.

In [ ]:
# Guided A1 — the projection, written once
def project(P, f):
    """P: (N, 3) points in metres (X right, Y up, Z forward). f in mm. Returns (N, 2) image points in mm."""
    P = np.asarray(P, float)
    return f * P[:, :2] / P[:, 2:3]

# a 1 m cube, centred straight ahead
corners = np.array([[x, y, z] for x in (-.5, .5) for y in (-.5, .5) for z in (-.5, .5)])
edges = [(i, j) for i in range(8) for j in range(i + 1, 8) if np.sum(corners[i] != corners[j]) == 1]

fig, ax = plt.subplots(1, 3, figsize=(10, 3.4))
for a, Z in zip(ax, (4, 8, 16)):
    xy = project(corners + [0, 0, Z], f=50)
    for i, j in edges:
        a.plot(*xy[[i, j]].T, color='teal')
    a.set_title(f'cube at Z = {Z} m'); a.set_xlim(-10, 10); a.set_ylim(-10, 10); a.set_aspect('equal')
    a.set_xlabel('x on the image (mm)')
plt.tight_layout(); plt.show()
for Z in (4, 8, 16):
    print(f'Z = {Z:2d} m → front face {50 * 1 / (Z - .5):.2f} mm wide')

Twice as far, half as big: the image size goes with $1/Z$.

In [ ]:
# Guided A2 — parallel rails meet at a vanishing point
Z = np.linspace(2, 300, 400)
fig, a = plt.subplots(figsize=(5, 3.4))
for X in (-0.75, 0.75):                       # two rails, 1.5 m apart, 1.5 m below the camera
    rail = np.stack([np.full_like(Z, X), np.full_like(Z, -1.5), Z], axis=1)
    a.plot(*project(rail, f=20).T, color='#444')
a.plot(0, 0, 'o', color='coral'); a.annotate('vanishing point', (0, 0), (2, 2))
a.set_aspect('equal'); a.set_xlim(-10, 10); a.set_ylim(-16, 4); a.set_title('rails seen by a 20 mm lens')
plt.show()
print('gap at  10 m:', 20 * 1.5 / 10, 'mm  ·  gap at 100 m:', 20 * 1.5 / 100, 'mm')

### Your turn A

In [ ]:
# 1. A car 1.6 m tall stands 20 m away; the lens has f = 35 mm. How tall is the car on the image, in mm?
car_height_mm = None
check('A1', car_height_mm, 2.8)

# 2. A person 1.8 m tall appears 9 mm tall through a 50 mm lens. How far away are they, in m?
person_distance_m = None
check('A2', person_distance_m, 10)

## B · Lenses

A sensor $w$ mm wide behind a lens of focal length $f$ sees
$$\text{FOV} = 2\arctan\frac{w}{2f}.$$

In [ ]:
# Guided B1 — field of view for a 36 mm wide sensor
def fov(f, w=36):
    return np.degrees(2 * np.arctan(w / (2 * f)))

for f in (12, 18, 24, 35, 50, 85, 200):
    print(f'f = {f:3d} mm → {fov(f):5.1f}°')

In [ ]:
# Guided B2 — zooming = keeping the centre. Treat the photo as taken at 24 mm.
def zoom(img, f, f0=24):
    h, w = img.shape[:2]; k = f0 / f
    ch, cw = int(h * k), int(w * k)
    crop = img[(h - ch) // 2:(h + ch) // 2, (w - cw) // 2:(w + cw) // 2]
    return cv2.resize(crop, (w, h), interpolation=cv2.INTER_LINEAR)

show([zoom(astronaut, f) for f in (24, 50, 100)], [f'f = {f} mm ({fov(f):.0f}°)' for f in (24, 50, 100)], cmap=None)

In [ ]:
# Guided B3 — blur circle of a thin lens (f = 50 mm) focused at s = 2 m:  c = A·f·|z − s| / (z·(s − f)),  A = f / N
def blur_circle_mm(z_mm, N, f=50, s=2000):
    A = f / N
    return A * f * np.abs(z_mm - s) / (z_mm * (s - f))

z = np.linspace(500, 10000, 300)
for N in (1.8, 4, 16):
    plt.plot(z / 1000, blur_circle_mm(z, N), label=f'f/{N}')
plt.axvline(2, color='gray', ls=':'); plt.xlabel('object distance (m)'); plt.ylabel('blur circle (mm)')
plt.legend(); plt.title('smaller aperture (larger N) → smaller blur → more depth of field'); plt.show()

### Your turn B

In [ ]:
# 1. Which focal length gives a 60° field of view on a 36 mm sensor? (in mm — use the half-angle)
f_for_60_mm = None
check('B1', f_for_60_mm, 31.18)

# 2. Focused at 2 m with f/2.8, how large is the blur circle of an object 1 m away? (in mm — use blur_circle_mm)
blur_1m_mm = None
check('B2', blur_1m_mm, 0.458)

## C · Sensors

To the computer an image is an array: height × width (× 3 for colour) numbers, each 0–255 in 8 bits.

In [ ]:
# Guided C1 — the image is an array
gray = cv2.cvtColor(astronaut, cv2.COLOR_RGB2GRAY)
print('shape', gray.shape, '· dtype', gray.dtype, '· min', gray.min(), '· max', gray.max())
print('a 5 × 5 patch:\n', gray[100:105, 200:205])

small = [cv2.resize(gray, (n, n), interpolation=cv2.INTER_AREA) for n in (128, 32, 16)]
show([cv2.resize(s, (512, 512), interpolation=cv2.INTER_NEAREST) for s in small], ['128 × 128', '32 × 32', '16 × 16'])

In [ ]:
# Guided C2 — bit depth: b bits keep 2**b levels
def quantise(img, bits):
    L = 2 ** bits
    return (np.round(img / 255 * (L - 1)) * (255 / (L - 1))).astype(np.uint8)

ramp = np.tile(np.linspace(0, 255, 512), (60, 1)).astype(np.uint8)
show([quantise(ramp, b) for b in (8, 4, 2)], ['8 bits', '4 bits — bands', '2 bits'], size=3.4)
show([quantise(gray, b) for b in (8, 3, 1)], ['8 bits', '3 bits', '1 bit'])

In [ ]:
# Guided C3 — shot noise: a pixel expecting N photons gets Poisson(N) of them, so SNR ≈ √N
def photon_image(img, N):
    """Simulate an exposure where a white pixel expects N photons."""
    counts = rng.poisson(N * (img.astype(float) / 255.0))   # float first: N × uint8 would overflow
    return np.clip(counts / N * 255, 0, 255).astype(np.uint8)

def measure_snr(N, size=200_000):
    """Mean ÷ standard deviation of a flat white patch."""
    counts = rng.poisson(N, size)
    return counts.mean() / counts.std()

show([photon_image(gray, N) for N in (4, 64, 1024)], ['4 photons', '64 photons', '1024 photons'])
for N in (16, 100, 1600):
    print(f'N = {N:5d} → measured SNR {measure_snr(N):5.1f}   (√N = {np.sqrt(N):.1f})')

In [ ]:
# Guided C4 — dynamic range: a room (2–20) with a window (200–2000) is 1000 : 1; 8 bits hold 255 : 1
H, W = 200, 320
y = np.linspace(0, 1, H)[:, None]; x = np.linspace(0, 1, W)[None, :]
scene = 2 + 18 * (1 - y) * np.ones_like(x)                              # the room
win = (x > .6) & (x < .9) & (y > .15) & (y < .6)
scene = np.where(win, 200 + 1800 * (1 - (y - .15) / .45), scene)        # the window
for ev in (-6, 0, 2):
    v = scene * 2.0 ** ev * 255 / 40
    print(f'exposure {ev:+d}: pure white {np.mean(v >= 255):5.1%} · pure black {np.mean(v < 1):5.1%}')

### Your turn C

In [ ]:
# 1. How many bytes does a 1920 × 1080 colour image take at 8 bits per channel, before compression?
image_bytes = None
check('C1', image_bytes, 6_220_800)

# 2. Measure the SNR of a white pixel that expects 400 photons (use measure_snr).
snr_400 = None
check('C2', snr_400, 20, tol=0.05)

## D · Colour

A colour pixel is three numbers (R, G, B). A single-chip sensor measures only **one** of them per photosite, through a Bayer filter: a repeating 2 × 2 tile **R G / G B**.

In [ ]:
# Guided D1 — three channels, three grey images
show([astronaut[..., c] for c in range(3)], ['R', 'G', 'B'])

In [ ]:
# Guided D2 — what the sensor records: an RGGB mosaic
def green_mask(h, w):
    m = np.zeros((h, w), bool); m[0::2, 1::2] = True; m[1::2, 0::2] = True
    return m

def mosaic(rgb):
    h, w, _ = rgb.shape
    m = np.zeros((h, w), np.uint8)
    m[0::2, 0::2] = rgb[0::2, 0::2, 0]          # R
    m[green_mask(h, w)] = rgb[..., 1][green_mask(h, w)]   # G (twice per tile)
    m[1::2, 1::2] = rgb[1::2, 1::2, 2]          # B
    return m

raw = mosaic(astronaut)
colour_view = np.zeros_like(astronaut)
colour_view[0::2, 0::2, 0] = raw[0::2, 0::2]; colour_view[1::2, 1::2, 2] = raw[1::2, 1::2]
colour_view[..., 1][green_mask(*raw.shape)] = raw[green_mask(*raw.shape)]
show([colour_view[100:164, 180:244]], ['the mosaic (a 64 × 64 corner, enlarged)'], cmap=None, size=4)

In [ ]:
# Guided D3 — demosaicing: guess the two missing colours from the neighbours
# (a) our own: average of the nearest neighbours of that colour (bilinear)
def demosaic_bilinear(raw):
    h, w = raw.shape; r = raw.astype(float)
    R = np.zeros((h, w)); G = np.zeros((h, w)); B = np.zeros((h, w))
    R[0::2, 0::2] = r[0::2, 0::2]; B[1::2, 1::2] = r[1::2, 1::2]; G[green_mask(h, w)] = r[green_mask(h, w)]
    kG = np.array([[0, 1, 0], [1, 4, 1], [0, 1, 0]]) / 4
    kRB = np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]]) / 4
    out = np.dstack([cv2.filter2D(R, -1, kRB), cv2.filter2D(G, -1, kG), cv2.filter2D(B, -1, kRB)])
    return np.clip(out, 0, 255).astype(np.uint8)

# (b) OpenCV. It names a Bayer pattern after the 2 × 2 block that starts at row 1, column 1 —
#     so our RGGB mosaic is 'BG' to OpenCV.
ours = demosaic_bilinear(raw)
opencv = cv2.cvtColor(raw, cv2.COLOR_BayerBG2RGB)
err = lambda a: np.abs(a.astype(int) - astronaut.astype(int)).mean()
print(f'mean error · ours {err(ours):.2f} · OpenCV {err(opencv):.2f}   (0 would be perfect)')
show([astronaut[150:214, 220:284], ours[150:214, 220:284], opencv[150:214, 220:284]], ['original', 'ours', 'OpenCV'], cmap=None)

In [ ]:
# Guided D4 — the same colours in HSV; OpenCV stores hue as 0–179 (degrees ÷ 2) in 8 bits
hsv = cv2.cvtColor(astronaut, cv2.COLOR_RGB2HSV)
turned = hsv.copy(); turned[..., 0] = (hsv[..., 0].astype(int) + 60) % 180      # +120°
show([astronaut, cv2.cvtColor(turned, cv2.COLOR_HSV2RGB)], ['original', 'hue + 120°'], cmap=None, size=3.6)

### Your turn D

In [ ]:
# 1. What fraction of the photosites in a Bayer mosaic measure green? (use green_mask on any even size)
green_fraction = None
check('D1', green_fraction, 0.5)

# 2. A 4000 × 3000 Bayer sensor: how many photosites measure green?
green_photosites = None
check('D2', green_photosites, 6_000_000)

---
**Done.** Back to the slides: [session 2 deck](https://classes.incortx.com/ComputerVision/session-02/slides.html) · more exam-style questions: [practice bank](https://classes.incortx.com/ComputerVision/practice/#s2).